# JDAR SFT → inspection → DPO

This notebook carries one model through the full experiment: reproduce the SFT run, inspect the resulting adapter, continue from that in-memory SFT adapter with DPO, then inspect the final adapter on exactly the same prompts.

The SFT hyperparameters below are copied from `sft_run_env/jdar-sft-finetuning.ipynb`. Change only the model configuration cell when repeating the experiment for either of the other two models. The source notebook did not retain those two model IDs.

Before running, attach both datasets and confirm `DPO_DATASET_PATH`. The default inspection prompts come from the training set only as a pipeline smoke test; replace them with held-out clauses before treating the comparison as an evaluation.

In [1]:
%%capture
# Kept from the successful SFT notebook so SFT and DPO use one compatible runtime.
!pip3 install trl
!pip3 install unsloth==2026.7.5
!pip3 install accelerate
!pip3 install wandb
!pip install -U "transformers>=5.10.4"
!pip install "unsloth_zoo" --upgrade --force-reinstall --no-deps

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # debugging only
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
from unsloth import FastLanguageModel, get_chat_template

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
from datasets import load_dataset
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import torch
import transformers
import wandb
from accelerate import PartialState
from huggingface_hub import login
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

In [7]:
print(f"transformers={transformers.__version__}")
import trl
print(f"trl={trl.__version__}")
print(f"torch={torch.__version__}")

transformers=5.14.1
trl=0.24.0
torch=2.10.0+cu128


## Credentials and experiment configuration

The source SFT notebook used the Kaggle secrets `HF_READ_DATASETS_TOKEN` and `WANDB_API_KEY`. A read token is sufficient for gated-model access and dataset downloads; use a write-capable token only if you later add Hub publishing.

In [8]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_READ_DATASETS_TOKEN")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_SILENT"] = "true"
login(token=HF_TOKEN)
wandb.login(key=WANDB_API_KEY, relogin=True)

True

In [9]:
MODEL_NAME = "unsloth/Ministral-3-14B-Base-2512"
CHAT_TEMPLATE = "mistral"
MAX_SEQUENCE_LENGTH = 2048
LOAD_IN_4BIT = True

SFT_DATASET_PATH = "/kaggle/input/datasets/adrinorosario/sft-dpo-run-json-coupled/sft_dataset_revised.json"
DPO_DATASET_PATH = "/kaggle/input/datasets/adrinorosario/sft-dpo-run-json-coupled/dpo_dataset_balanced.json"

WORKING_DIR = Path("/kaggle/working")
SFT_OUTPUT_DIR = str(WORKING_DIR / "minisstral_jdar_sft")
DPO_OUTPUT_DIR = str(WORKING_DIR / "ministral_jdar_dpo")
SFT_METRICS_PATH = WORKING_DIR / "ministral_jdar_sft_metrics.csv"
DPO_METRICS_PATH = WORKING_DIR / "ministral_jdar_dpo_metrics.csv"

In [10]:
def get_available_gpu_ids() -> list[int]:
    """Detect visible CUDA devices without initializing a CUDA context."""
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        )
        gpu_ids = [int(x.strip()) for x in result.stdout.strip().splitlines() if x.strip()]
        if not gpu_ids:
            raise ValueError("nvidia-smi returned no GPUs")
        return gpu_ids
    except Exception as error:
        print(f"nvidia-smi detection failed ({error}); falling back to torch.")
        return list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []

AVAILABLE_GPUS = get_available_gpu_ids()
assert AVAILABLE_GPUS, "A CUDA GPU is required for this 12B QLoRA run."
device_string = f"cuda:{PartialState().local_process_index}"
print(f"Detected GPUs: {AVAILABLE_GPUS}; this process uses {device_string}")
print(f"Device name: {torch.cuda.get_device_name(0)}")

Detected GPUs: [0, 1]; this process uses cuda:0
Device name: Tesla T4


In [11]:
print("GPU count:", torch.cuda.device_count())
!nvidia-smi

GPU count: 2
Thu Jul 30 02:12:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             29W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------------

## SFT setup and training

This section preserves the source run’s LoRA targets and SFT hyperparameters. The extra validation makes a missing or wrongly attached dataset fail early.

In [12]:
def load_sft_dataset():
    dataset = load_dataset("json", data_files=SFT_DATASET_PATH)
    required_columns = {"prompt", "response"}
    missing = required_columns - set(dataset["train"].column_names)
    if missing:
        raise ValueError(f"SFT data is missing columns: {sorted(missing)}")
    print(f"SFT rows: {len(dataset['train']):,}; columns: {dataset['train'].column_names}")
    return dataset

def load_model_tokenizer(model_name, maximum_sequence_length, load_in_4_bit, chat_template_name):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=maximum_sequence_length,
        load_in_4bit=load_in_4_bit,
        device_map="balanced",
        max_memory={0: "13GiB", 1: "13GiB"},
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    tokenizer = get_chat_template(tokenizer, chat_template=chat_template_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

In [13]:
def formatting_prompts_func(batch, tokenizer):
    texts = []
    for prompt, response in zip(batch["prompt"], batch["response"]):
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
        texts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, reasoning_effort="high"
            )
        )
    return {"text": texts}

def map_text_column(example):
    return example["text"]

def setup_wandb_run_logging(project_name, run_name):
    return wandb.init(project=project_name, name=run_name)

def save_training_metrics(trainer, output_path):
    pd.DataFrame(trainer.state.log_history).to_csv(output_path, index=False)
    print(f"Saved metrics to {output_path}")

In [14]:
def sft_training_setup():
    raw_dataset = load_sft_dataset()
    model, tokenizer = load_model_tokenizer(
        model_name=MODEL_NAME,
        maximum_sequence_length=MAX_SEQUENCE_LENGTH,
        load_in_4_bit=LOAD_IN_4BIT,
        chat_template_name=CHAT_TEMPLATE,
    )
    formatted_dataset = raw_dataset.map(
        lambda batch: formatting_prompts_func(batch, tokenizer), batched=True
    )
    training_args = SFTConfig(
        output_dir=SFT_OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        warmup_steps=5,
        max_steps=120,
        learning_rate=2e-4,
        optim="adamw_8bit", 
        fp16=True,
        bf16=False,
        logging_steps=1,
        average_tokens_across_devices=False,
        max_length=MAX_SEQUENCE_LENGTH,
        ddp_find_unused_parameters=False,
        eval_strategy="no",
        eval_steps=10,
        per_device_eval_batch_size=4,
        report_to="wandb",
    )
    return raw_dataset, formatted_dataset, model, tokenizer, training_args

sft_raw_dataset, sft_dataset, model, tokenizer, sft_training_args = sft_training_setup()

Generating train split: 0 examples [00:00, ? examples/s]

SFT rows: 984; columns: ['prompt', 'response']
==((====))==  Unsloth 2026.7.5: Fast Ministral3 patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The tokenizer you are loading from 'unsloth/ministral-3-14b-base-2512-unsloth-bnb-4bit' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

In [15]:
def run_sft(dataset, model, tokenizer, formatting_function, training_args, run_name):
    setup_wandb_run_logging(project_name="SFT-DPO Runs", run_name=run_name)
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset["train"],
        formatting_func=formatting_function,
        args=training_args,
    )
    trainer.train()
    wandb.finish()
    return trainer

sft_trainer = run_sft(
    dataset=sft_dataset,
    model=model,
    tokenizer=tokenizer,
    formatting_function=map_text_column,
    training_args=sft_training_args,
    run_name=MODEL_NAME.replace("/", "_") + "-sft",
)
save_training_metrics(sft_trainer, SFT_METRICS_PATH)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/984 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 984 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 19,660,800 of 13,964,692,480 (0.14% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.920704
2,2.735947
3,2.827338
4,2.926749
5,2.617228
6,2.450665
7,2.327117
8,2.339105
9,2.300290
10,2.531022


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/minisstral_jdar_sft/checkpoint-120/tokenizer_config.json.


Saved metrics to /kaggle/working/ministral_jdar_sft_metrics.csv


In [ ]:
# only if OOM after SFT Training
from pathlib import Path

SFT_SAVE_DIR = Path("/kaggle/working/ministral_jdar_sft_adapter")
SFT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# In the combined notebook:
sft_trainer.save_model(str(SFT_SAVE_DIR))

# Save the matching chat template/tokenizer too.
tokenizer.save_pretrained(str(SFT_SAVE_DIR))

print("Saved post-SFT adapter and tokenizer:")
print(list(SFT_SAVE_DIR.iterdir()))

## Inspect the SFT adapter

Use the same prompt list again after DPO so the before/after table is directly comparable. The prefilled examples are deliberately labelled as a smoke test because they are from the training dataset.

In [16]:
# Replace these with held-out clauses for a meaningful quality comparison.
INSPECTION_PROMPTS = sft_raw_dataset["train"].select(range(min(3, len(sft_raw_dataset["train"]))))["prompt"]

def generate_response(model, tokenizer, prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(next(model.parameters()).device)
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0, input_ids.shape[-1]:], skip_special_tokens=True).strip()

def inspect_model_outputs(model, tokenizer, prompts, stage):
    FastLanguageModel.for_inference(model)
    rows = [
        {"stage": stage, "prompt": prompt, "response": generate_response(model, tokenizer, prompt)}
        for prompt in prompts
    ]
    return pd.DataFrame(rows)

In [17]:
sft_inspection = inspect_model_outputs(model, tokenizer, INSPECTION_PROMPTS, stage="after_sft")
display(sft_inspection)

# Switch Unsloth back to training mode before constructing DPOTrainer.
FastLanguageModel.for_training(model)

Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,stage,prompt,response
0,after_sft,Given the following contractual clause which f...,The parties’ agreement to arbitrate is contain...
1,after_sft,Given the following contractual clause which f...,The Court of Appeals held that the\n“[t]he par...
2,after_sft,Given the following contractual clause which f...,The parties’ agreement provides that the\n“[p]...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Mistral3ForConditionalGeneration(
      (model): Mistral3Model(
        (vision_tower): PixtralVisionModel(
          (patch_conv): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (ln_pre): PixtralRMSNorm((1024,), eps=1e-05)
          (transformer): PixtralTransformer(
            (layers): ModuleList(
              (0-23): 24 x PixtralAttentionLayer(
                (attention_norm): PixtralRMSNorm((1024,), eps=1e-05)
                (feed_forward): PixtralMLP(
                  (gate_proj): Linear(in_features=1024, out_features=4096, bias=False)
                  (up_proj): Linear(in_features=1024, out_features=4096, bias=False)
                  (down_proj): Linear(in_features=4096, out_features=1024, bias=False)
                  (act_fn): SiLUActivation()
                )
                (attention): PixtralAttention(
                  (k_proj): Linear(in_features=1024, out_features=1024, b

## DPO setup and training

The local DPO dataset already has the standard TRL preference columns: `prompt`, `chosen`, and `rejected`. Because the SFT run used a Gemma chat template, the prompt is converted to the same user-to-assistant generation prefix before DPO; the preferred and rejected completions remain separate for `DPOTrainer`.

`ref_model=None` intentionally tells TRL to use the SFT policy at the start of DPO as the reference, while the same LoRA adapter is updated as the policy. This avoids reinitializing from the base model.

In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
assert torch.cuda.device_count() == 2, "Kaggle is not exposing both T4 GPUs."

In [ ]:
# load the model only if it was saved after sft run
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/working/gptoss_jdar_sft_adapter", # Points to your local SFT save directory
    max_seq_length = MAX_SEQUENCE_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
    device_map="balanced",
    max_memory={
            0: "13GiB",
            1: "13GiB",
    },
)
print("Actual placement:", model.hf_device_map)

In [18]:
def load_dpo_dataset():
    dataset = load_dataset("json", data_files=DPO_DATASET_PATH)
    required_columns = {"prompt", "chosen", "rejected"}
    missing = required_columns - set(dataset["train"].column_names)
    if missing:
        raise ValueError(f"DPO data is missing columns: {sorted(missing)}")
    empty_counts = {name: sum(not value.strip() for value in dataset["train"][name]) for name in required_columns}
    if any(empty_counts.values()):
        raise ValueError(f"DPO data contains blank values: {empty_counts}")
    print(f"DPO rows: {len(dataset['train']):,}; columns: {dataset['train'].column_names}")
    return dataset

def prepare_dpo_dataset(dataset, tokenizer):
    tokenizer.padding_side = "left"
    eos = tokenizer.eos_token or ""

    def format_preference_batch(batch):
        prompts = []
        for prompt in batch["prompt"]:
            messages = [{"role": "user", "content": prompt}]
            prompts.append(
                tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, reasoning_effort="high"
                )
            )
        return {
            "prompt": prompts,
            "chosen": [text if not eos or text.endswith(eos) else text + eos for text in batch["chosen"]],
            "rejected": [text if not eos or text.endswith(eos) else text + eos for text in batch["rejected"]],
        }

    return dataset.map(format_preference_batch, batched=True)

dpo_raw_dataset = load_dpo_dataset()
dpo_dataset = prepare_dpo_dataset(dpo_raw_dataset, tokenizer)
print(dpo_dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DPO rows: 984; columns: ['prompt', 'chosen', 'rejected']


Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

{'prompt': '<s>[INST] Given the following contractual clause which falls in the category of Non-Compete, provide judicial reasoning of the kind a court would apply when interpreting clauses that raise similar legal questions:\n\nUnless accepted by the Principal, the Distributor agrees that during the term of this Agreement, the Distributor, either directly or indirectly, shall handle no products that are competitive with the Products within the Territory. [/INST]', 'chosen': 'at 15–16.)\nAnd yet if that is the case, and the outgoing quality control provision was indeed limited to\naddressing defects that can be ascertained by inspection, then defendant is correct that “there\nwould be no reason for the language of [§] 981.42(b) to refer to ‘minimum quality and\ninspection requirements.’” (Def.’s Mot. at 33 (emphasis in the original).)\n23\n   Plaintiffs’ comparisons of the Salmonella Rule to terms and conditions applicable to other\ncommodities, raised only in their statement of facts 

In [19]:
dpo_training_args = DPOConfig(
    output_dir=DPO_OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    warmup_steps=10,
    max_steps=120,
    optim="adamw_8bit", 
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=1,
    average_tokens_across_devices=False,
    max_length=MAX_SEQUENCE_LENGTH,
    beta=0.1,
    ddp_find_unused_parameters=False,
    report_to="wandb",
    eval_strategy="no",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_training_args,
    train_dataset=dpo_dataset["train"],
    processing_class=tokenizer,
)

Extracting prompt in train dataset (num_proc=8):   0%|          | 0/984 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=8):   0%|          | 0/984 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/984 [00:00<?, ? examples/s]

In [ ]:
setup_wandb_run_logging(
    project_name="SFT-DPO Runs",
    run_name=MODEL_NAME.replace("/", "_") + "-dpo-from-sft",
)
dpo_trainer.train()
wandb.finish()
save_training_metrics(dpo_trainer, DPO_METRICS_PATH)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 984 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 19,660,800 of 13,964,692,480 (0.14% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
1,1.482926,6.233635,4.993576,0.625000,1.240059,-445.326141,-374.260193,-0.726139,-0.874945
2,0.592044,6.654299,4.604496,0.875000,2.049803,-389.340698,-344.303864,-0.735726,-1.036009
3,1.310387,4.159015,4.187303,0.625000,-0.028288,-355.567200,-330.667389,-1.194737,-0.811625
4,0.972338,4.814435,4.319275,0.750000,0.495161,-366.810181,-366.919434,-0.769253,-1.028837
5,1.277593,5.616521,4.711705,0.750000,0.904816,-316.767517,-306.306396,-0.802052,-0.664989
6,1.584204,5.162246,5.546981,0.500000,-0.384735,-319.735565,-319.383606,-0.859068,-0.481539
7,1.885098,5.156963,6.021688,0.375000,-0.864725,-313.110352,-295.268524,-0.765057,-0.472892
8,1.163403,4.293653,4.379164,0.500000,-0.085511,-345.068542,-392.391327,-0.743449,-0.896825
9,0.571252,4.672463,3.288687,0.750000,1.383777,-360.637695,-370.291748,-0.778518,-0.923386
10,0.098075,5.486659,1.701035,1.000000,3.785624,-344.673309,-332.313141,-0.647387,-0.839556


## Inspect and save the DPO adapter

The comparison table preserves both response sets for the same prompts. Review it qualitatively; it is not a substitute for a held-out, category-stratified evaluation.

In [21]:
dpo_inspection = inspect_model_outputs(dpo_trainer.model, tokenizer, INSPECTION_PROMPTS, stage="after_dpo")
comparison = sft_inspection.rename(columns={"response": "sft_response"}).drop(columns="stage")
comparison["dpo_response"] = dpo_inspection["response"]
display(comparison)

comparison_path = WORKING_DIR / "jdar_sft_vs_dpo_responses.csv"
comparison.to_csv(comparison_path, index=False)
print(f"Saved response comparison to {comparison_path}")

Both `max_new_tokens` (=512) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [22]:
FINAL_ADAPTER_DIR = WORKING_DIR / "ministral_jdar_dpo_adapter_2"
dpo_trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
print(f"Saved final DPO adapter and tokenizer to {FINAL_ADAPTER_DIR}")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/ministral_jdar_dpo_adapter_2/tokenizer_config.json.


Saved final DPO adapter and tokenizer to /kaggle/working/ministral_jdar_dpo_adapter_2


In [31]:
import os
import shutil
from IPython.display import FileLink

# 1. Define base directory and target folder name
base_dir = '/kaggle/working'
folder_name = '/kaggle/working/ministral_jdar_dpo_adapter_2'

# 2. Set the absolute paths
folder_to_zip = os.path.join(base_dir, folder_name)
output_zip_path = os.path.join(base_dir, folder_name)

# 3. Create the zip file inside /kaggle/working
# shutil automatically appends '.zip' to the output path
shutil.make_archive(output_zip_path, 'zip', folder_to_zip)

# 4. Generate the clickable download link
FileLink(f'{folder_name}.zip')

/kaggle/working/ministral_jdar_dpo_adapter_2.zip